# Justificación de `shared/concepts.py`

`CONCEPTS` asigna a cada concepto contable (ingresos, activo, etc.) una lista de tags XBRL en orden de prioridad. En cada período, el concepto toma el valor del **primer tag de la lista que tiene valor**. Este notebook justifica esas listas con datos, y también cada número de `docs/ratios.md` que no es una cobertura por tag.

**Qué responde cada sección:**

| Sección | Pregunta |
|---|---|
| Prueba 1 | ¿La línea que la empresa llama, por ejemplo, "Total assets" usa un tag de la lista? |
| Prueba 2 | ¿Cada tag de la lista aporta empresas o períodos? |
| Prueba 3 | Cuando dos tags de la lista dan valores distintos, ¿el primero es el total que presenta la empresa? |
| Prueba 4 | ¿Sirven los tags alternativos de la guía de la profesora? |
| Prueba 5 | ¿Las construcciones (un concepto calculado con otros tags) dan el mismo valor que el tag? |
| Sección 6 | Números de los ratios descartados. |
| Sección 7 | El caso de HCA Healthcare (patrimonio negativo). |

**Vocabulario del notebook.**
- **Período:** una empresa, una fecha de fin y una duración: (`cik`, `ddate`, `qtrs`).
- **Difieren / coinciden:** dos valores a y b difieren si |a − b| > 1% de |a|. Si no, coinciden.
- **Conflicto:** un período con dos o más tags del mismo concepto que difieren. Solo en un conflicto el orden de la lista cambia el valor del concepto.

**Datos.** `data/num.csv` (números de 2018–2023; solo trae valores en USD), `data/sub.csv` (documentos) y `pre.txt` de cada zip (líneas de los estados contables). Al final, la sección **Resultado** lista cada número de `docs/ratios.md` con la celda que lo calcula.

## Preparación

La celda siguiente carga los datos con `ddate` entre 2018 y 2023. `num.csv` ya trae solo valores en USD (`build_dataset.ipynb`). Define:
- `num`: una fila por número (tag, período, valor), con el sector de la empresa.
- `companies`: una fila por empresa, con el nombre, el sector y el SIC de su documento más reciente.
- `K`: las tres columnas que definen un período.
- `mapa`: `CONCEPTS` como tabla, con la posición de cada tag en su lista (`priority`: 0 es el primero).

La salida es la cantidad de empresas por sector.

In [1]:
import glob
import zipfile

import pandas as pd

from shared.concepts import CONCEPTS

# numbers: 2018-2023; num.csv only has USD values
num = pd.read_csv("data/num.csv", dtype={"cik": str})
num = num[num["ddate"].between(20180101, 20231231)]

# documents; each company gets the sector and name of its most recent document
sub = pd.read_csv("data/sub.csv", dtype={"cik": str})
companies = sub.drop_duplicates("cik", keep="last")[["cik", "name", "sector", "sic"]]
num = num.merge(companies[["cik", "sector"]], on="cik")

# a period: company, end date and duration in quarters
K = ["cik", "ddate", "qtrs"]

# CONCEPTS as a table: one row per concept and tag, with its position in the list
mapa = pd.DataFrame(
    [(c, t, p) for c, tags in CONCEPTS.items() for p, t in enumerate(tags)],
    columns=["concept", "tag", "priority"],
)
mapa
# num.groupby("sector")["cik"].nunique()


,concept,tag,priority
0,revenue,Revenues,0
1,revenue,RevenueFromContractWithCustomerExcludingAssess...,1
2,revenue,RevenueFromContractWithCustomerIncludingAssess...,2
3,revenue,SalesRevenueNet,3
4,revenue,SalesRevenueServicesNet,4
5,revenue,SalesRevenueGoodsNet,5
6,revenue,HealthCareOrganizationRevenue,6
7,revenue,HealthCareOrganizationPatientServiceRevenue,7
8,revenue,OilAndGasRevenue,8
9,revenue,OilAndGasSalesRevenue,9


`values` es la misma información de `num` en formato ancho: una fila por período y una columna por tag. Así, una cuenta entre tags es una resta entre columnas. Ejemplo: `col("LiabilitiesAndStockholdersEquity") - col("StockholdersEquity")`.

Dos funciones auxiliares:
- `col(tag)`: la columna de un tag. Si ninguna empresa lo reporta, devuelve una columna vacía (todo NaN) en lugar de un error.
- `first(tags)`: en cada período, el valor del primer tag de la lista que tiene valor. Es la regla de `CONCEPTS`.

La salida es (cantidad de períodos, cantidad de tags).

In [2]:
# one row per period and one column per tag; NaN = the company did not report that tag
values = num.pivot_table(index=K, columns="tag", values="value", aggfunc="first")
print("shape:", values.shape)
values.head()


def col(tag: str) -> pd.Series:
    """Values of one tag in every period (all NaN if no company reports it)."""
    return values[tag] if tag in values else pd.Series(index=values.index, dtype=float)


def first(tags: list[str]) -> pd.Series:
    """In each period, the value of the first tag of the list that has one."""
    out = pd.Series(index=values.index, dtype=float)
    for tag in tags:
        out = out.fillna(col(tag))
    return out


shape: (19598, 2294)


## Prueba 1: ¿qué tag usa la línea del concepto?

**Idea.** Si la lista es correcta, la línea que la empresa titula, por ejemplo, "Total assets" tiene que usar un tag de la lista.

`pre.txt` tiene una fila por línea de cada estado contable presentado. Columnas que se usan:
- `stmt`: el estado. `IS` es el estado de resultados, `BS` el balance y `CF` el estado de flujo de efectivo.
- `inpth`: 0 si es el valor principal, 1 si es un valor a modo de aclaracion.
- `line`: la posición de la línea dentro del estado (1 es la primera).
- `plabel`: el texto de la línea, tal como lo ve el lector.
- `version`: la taxonomía del tag. Si no empieza con `us-gaap`, es un tag propio de la empresa y se reemplaza por "(tag propio)".

La celda lee las líneas de los tres estados en los documentos de `sub.csv`. Tarda unos 20 segundos. La salida muestra las primeras filas.

In [3]:
parts = []

for path in sorted(glob.glob("zips/*.zip")):
    with zipfile.ZipFile(path) as z:
        p = pd.read_csv(
            z.open("pre.txt"),
            sep="\t",
            dtype=str,
            usecols=["adsh", "stmt", "inpth", "line", "tag", "version", "plabel"],
        )
    # keep our documents, the three main statements and the lines outside parentheses
    keep = (
        p["adsh"].isin(sub["adsh"])
        & p["stmt"].isin(["IS", "BS", "CF"])
        & (p["inpth"] == "0")
    )
    parts.append(p[keep])


pre = pd.concat(parts, ignore_index=True)

pre["line"] = pre["line"].astype(int)

# line text in lower case, with the typographic apostrophe replaced by the plain one
pre["label"] = pre["plabel"].str.lower().str.replace("’", "'").str.strip()

# a tag outside the us-gaap taxonomy is the company's own tag
pre.loc[~pre["version"].str.startswith("us-gaap"), "tag"] = "(tag propio)"


# pre[["adsh", "stmt", "line", "tag", "plabel"]].head()


In [4]:
pre.head()

,adsh,line,stmt,inpth,tag,version,plabel,label
0,0000007332-18-000016,2,IS,0,(tag propio),0000007332-18-000016,Gas sales,gas sales
1,0000007332-18-000016,3,IS,0,(tag propio),0000007332-18-000016,Oil sales,oil sales
2,0000007332-18-000016,4,IS,0,(tag propio),0000007332-18-000016,NGL sales,ngl sales
3,0000007332-18-000016,5,IS,0,RefiningAndMarketingRevenue,us-gaap/2017,Marketing,marketing
4,0000007332-18-000016,6,IS,0,GasGatheringTransportationMarketingAndProcessi...,us-gaap/2017,Gas gathering,gas gathering


`LABELS` define, para cada concepto, su estado y una **expresión regular** con los textos de línea que lo representan. Cómo se leen:
- `(total )?`: el texto "total " puede estar o no.
- `a|b`: el texto es `a` o `b`.
- `revenues?`: "revenue" o "revenues".
- `\(` y `\)`: un paréntesis literal, como en "income (loss)".
- `(?!.*noncontrolling)`: "que no contenga noncontrolling".
- `fullmatch`: la expresión tiene que cubrir el texto completo, no una parte.

Ejemplo: para el activo, la expresión `total assets` encuentra la línea "Total assets" y no encuentra "Total current assets".

**Resultado de la celda**, para cada concepto:
- **documentos:** documentos con al menos una línea cuyo texto coincide con la expresión.
- **tag de la lista en esa línea (%):** documentos en que esa línea usa un tag de la lista.
- **tag de la lista en el estado (%):** documentos en que un tag de la lista está en alguna línea del mismo estado, aunque su texto sea otro. Ejemplo: "Total XYZ Corp. stockholders' equity" no coincide con la expresión, pero es el patrimonio.

PRECAUCIÓN: las expresiones cubren los textos más comunes, no todos. La prueba mide esos documentos.

In [5]:
# the amount part of a result line: "income", "loss", "income (loss)", ...
AMOUNT = r"(income|loss|earnings|income \(loss\)|\(loss\) income|earnings \(loss\))"

# concept -> (statement, regular expression for the line text)
LABELS = {
    "revenue": (
        "IS",
        r"(total )?(net )?(operating )?(revenues?|sales)(, net)?( and other( income)?)?",
    ),
    "operating_income": (
        "IS",
        rf"(total )?operating {AMOUNT}|{AMOUNT} from operations",
    ),
    # "net income" or "net income attributable to X", except when X are the minority partners
    "net_income": (
        "IS",
        rf"net {AMOUNT}( attributable to (?!.*(noncontrolling|non-controlling|minority)).*)?",
    ),
    "assets": ("BS", r"total assets"),
    "liabilities": ("BS", r"total liabilities"),
    "equity": (
        "BS",
        r"total (stockholders|shareholders|partners|members)'? ?(equity|capital|deficit|equity \(deficit\)|\(deficit\) equity)",
    ),
    "current_assets": ("BS", r"total current assets"),
    "current_liabilities": ("BS", r"total current liabilities"),
    "operating_cash_flow": (
        "CF",
        r"net cash (provided by|used in|provided by \(used in\)|\(used in\) provided by|from) operating activities",
    ),
}


def label_lines(concept: str) -> pd.DataFrame:
    """Lines of the concept's statement whose whole text matches the concept's expression."""
    stmt, pattern = LABELS[concept]

    _filter = (
        (pre["stmt"] == stmt)  # filtramos por statement (IS, BS o CF)
        & pre["label"].str.fullmatch(pattern)  # filtramos por patron
    )

    return pre[_filter]


rows = []
for concept, tags in CONCEPTS.items():
    lines = label_lines(concept)

    # per document: does some line with the concept's text use a tag of the list?
    in_line = lines["tag"].isin(tags).groupby(lines["adsh"]).any()

    # per document: is a tag of the list anywhere in the same statement?
    statement = pre[
        # filtramos por statement (IS, BS o CF)
        (pre["stmt"] == LABELS[concept][0])
        # filtramos por documentos que tienen la linea del concepto
        & pre["adsh"].isin(lines["adsh"])
    ]

    in_statement = statement["tag"].isin(tags).groupby(statement["adsh"]).any()

    rows.append(
        {
            "concepto": concept,
            "documentos": len(in_line),
            "tag de la lista en esa línea (%)": round(in_line.mean() * 100, 1),
            "tag de la lista en el estado (%)": round(in_statement.mean() * 100, 1),
        }
    )
pd.DataFrame(rows).set_index("concepto")

,documentos,tag de la lista en esa línea (%),tag de la lista en el estado (%)
concepto,,,
revenue,6393,98.9,99.6
operating_income,6347,99.0,99.1
net_income,7462,90.4,91.6
assets,7937,100.0,100.0
liabilities,6156,99.8,99.9
equity,6399,84.6,95.4
current_assets,7798,100.0,100.0
current_liabilities,7792,99.9,99.9
operating_cash_flow,6998,99.7,100.0


**¿Qué tags usan los documentos que fallan la prueba?** `outside(concepto)` lista los tags de las líneas del concepto en los documentos donde ninguna de esas líneas usa un tag de la lista. Sirve para encontrar tags que faltan. Para ver otro concepto, cambiar el argumento.

Para `net_income`, los tags que aparecen son conceptos distintos de la ganancia de los accionistas de la empresa:
- `ProfitLoss`: incluye la parte de los socios minoritarios (prueba 4).
- `NetIncomeLossAvailableToCommonStockholdersBasic`: la ganancia después de pagar los dividendos de las acciones preferidas.

In [6]:
def outside(concept: str, n: int = 8) -> pd.Series:
    """Tags of the concept's lines in the documents where no such line uses a tag of the list."""
    lines = label_lines(concept)
    has = lines["tag"].isin(CONCEPTS[concept]).groupby(lines["adsh"]).transform("any")
    # count each tag once per document
    return lines[~has].drop_duplicates(["adsh", "tag"])["tag"].value_counts().head(n)


outside("net_income")

tag
ProfitLoss                                                          471
NetIncomeLossAvailableToCommonStockholdersBasic                     344
(tag propio)                                                         72
EarningsPerShareBasic                                                22
EarningsPerShareDiluted                                              19
NetIncomeLossAvailableToCommonStockholdersDiluted                    16
NetIncomeLossPerOutstandingLimitedPartnershipUnitBasicNetOfTax       14
NetIncomeLossNetOfTaxPerOutstandingLimitedPartnershipUnitDiluted     14
Name: count, dtype: int64

## Prueba 2: ¿cuánto aporta cada tag?

Para cada tag, en el orden de la lista:
- **empresas energía / salud:** empresas que reportan el tag al menos una vez.
- **nuevas energía / salud:** empresas que reportan el tag y **ninguno de los anteriores** de la lista. Es lo que la lista pierde si se saca el tag.
- **períodos en que se elige:** períodos en que el tag es el primero de la lista con valor. Es el uso real del tag.

Un tag con 0 períodos no cambia ningún valor: sacarlo de la lista no tiene efecto. Un tag con pocas empresas nuevas pero muchos períodos cubre huecos de empresas que ya tienen otro tag en otros períodos.

In [7]:
# tag chosen in each period: the first one of the list with a value
chosen = (
    num.merge(mapa, on="tag")
    .sort_values("priority")
    .drop_duplicates(["concept", *K])
    .groupby(["concept", "tag"])
    .size()
)

rows = []
for concept, tags in CONCEPTS.items():
    seen = set()  # companies covered by the previous tags of the list
    for tag in tags:
        ciks = num.loc[num["tag"] == tag, ["cik", "sector"]].drop_duplicates()
        total = ciks["sector"].value_counts()
        new = ciks.loc[~ciks["cik"].isin(seen), "sector"].value_counts()
        seen |= set(ciks["cik"])
        rows.append(
            {
                "concepto": concept,
                "tag": tag,
                "empresas energía": total.get("energy", 0),
                "empresas salud": total.get("health", 0),
                "nuevas energía": new.get("energy", 0),
                "nuevas salud": new.get("health", 0),
                "períodos en que se elige": chosen.get((concept, tag), 0),
            }
        )
pd.DataFrame(rows).set_index(["concepto", "tag"])

empresas energía  \
concepto            tag                                                                    
revenue             Revenues                                                         213   
                    RevenueFromContractWithCustomerExcludingAssesse...               126   
                    RevenueFromContractWithCustomerIncludingAssesse...                69   
                    SalesRevenueNet                                                   26   
                    SalesRevenueServicesNet                                           10   
                    SalesRevenueGoodsNet                                              11   
                    HealthCareOrganizationRevenue                                      0   
                    HealthCareOrganizationPatientServiceRevenue                        0   
                    OilAndGasRevenue                                                  43   
                    OilAndGasSalesRevenue                                             52   
operating_income    OperatingIncomeLoss                                              266   
net_income          NetIncomeLoss                                                    299   
assets              Assets                                                           306   
liabilities         Liabilities                                                      236   
equity              StockholdersEquity                                               267   
                    PartnersCapital                                                   38   
                    MembersEquity                                                     12   
current_assets      AssetsCurrent                                                    298   
current_liabilities LiabilitiesCurrent                                               300   
operating_cash_flow NetCashProvidedByUsedInOperatingActivities                       303   
                    NetCashProvidedByUsedInOperatingActivitiesConti...                73   

                                                                        empresas salud  \
concepto            tag                                                                  
revenue             Revenues                                                        88   
                    RevenueFromContractWithCustomerExcludingAssesse...              97   
                    RevenueFromContractWithCustomerIncludingAssesse...              28   
                    SalesRevenueNet                                                 12   
                    SalesRevenueServicesNet                                         13   
                    SalesRevenueGoodsNet                                             3   
                    HealthCareOrganizationRevenue                                    9   
                    HealthCareOrganizationPatientServiceRevenue                      9   
                    OilAndGasRevenue                                                 0   
                    OilAndGasSalesRevenue                                            0   
operating_income    OperatingIncomeLoss                                            162   
net_income          NetIncomeLoss                                                  173   
assets              Assets                                                         175   
liabilities         Liabilities                                                    157   
equity              StockholdersEquity                                             171   
                    PartnersCapital                                                  0   
                    MembersEquity                                                    6   
current_assets      AssetsCurrent                                                  173   
current_liabilities LiabilitiesCurrent                                             175   
operating_cash_flow NetCashProvidedByUsedInOperatingActivities   

## Prueba 3: ¿el orden elige el total?

**Idea.** En un estado contable, el total está **después** de sus partes. Ejemplo: "Oil sales", "Gas sales" y después "Total revenues". Entonces, en un conflicto, el tag correcto es el de la **última línea** del estado. La prueba compara esa regla con la prioridad de la lista.

Se aplica a los tres conceptos con más de un tag. Para cada uno, la celda:
1. Busca los períodos con más de un tag de la lista.
2. Se queda con los conflictos: los valores difieren más de 1% (`relative_spread` > 0,01).
3. Se queda con los conflictos en que los dos valores vienen del mismo documento (`adsh`). Con la primera versión publicada (`docs/decisions.md`, sección 8), dos valores del mismo período pueden venir de documentos distintos, y sus líneas no se pueden comparar.
4. Busca la línea de cada tag en el estado del concepto (`pre`) y compara dos elecciones: la de la prioridad (`by_priority`) y la de la última línea (`by_last_line`).

La celda guarda los conflictos en `conflicts`, para las celdas siguientes. Después de la tabla, el detalle de ingresos y el de patrimonio.

In [8]:
STATEMENT = {c: s for c, (s, _) in LABELS.items()}


def relative_spread(v: pd.Series) -> float:
    """(largest - smallest) / largest absolute value; 0 when all values are equal."""
    return (v.max() - v.min()) / max(v.abs().max(), 1)


conflicts = {}  # concept -> rows of its conflicts, used again in the next cells
rows = []
for concept, tags in CONCEPTS.items():
    if len(tags) < 2:
        continue  # with one tag there is no order
    rows_concept = num.merge(mapa[mapa["concept"] == concept], on="tag")

    # 1. periods with more than one tag of the list
    several = rows_concept[rows_concept.groupby(K)["tag"].transform("nunique") > 1]

    # 2. conflicts: the values differ by more than 1%
    c = several[several.groupby(K)["value"].transform(relative_spread) > 0.01]
    conflicts[concept] = c

    # 3. conflicts with both values from one document, and both tags in the statement
    same_doc = c[c.groupby(K)["adsh"].transform("nunique") == 1]
    lines = pre.loc[pre["stmt"] == STATEMENT[concept], ["adsh", "tag", "line"]]
    same_doc = same_doc.merge(lines.drop_duplicates(["adsh", "tag"]), on=["adsh", "tag"])
    same_doc = same_doc[same_doc.groupby(K)["tag"].transform("nunique") > 1]

    # 4. the tag chosen by the priority and the tag of the last line
    by_priority = same_doc.sort_values("priority").drop_duplicates(K).set_index(K)["tag"]
    by_last_line = same_doc.sort_values("line").drop_duplicates(K, keep="last").set_index(K)["tag"]
    agree = by_priority == by_last_line.reindex(by_priority.index)

    rows.append(
        {
            "concepto": concept,
            "períodos con el concepto": rows_concept.groupby(K).ngroups,
            "con más de un tag": several.groupby(K).ngroups,
            "conflictos (difieren > 1%)": c.groupby(K).ngroups,
            "mismo documento, en el estado": len(by_priority),
            "prioridad = última línea (%)": round(agree.mean() * 100, 1),
        }
    )
pd.DataFrame(rows).set_index("concepto")

,períodos con el concepto,con más de un tag,conflictos (difieren > 1%),"mismo documento, en el estado",prioridad = última línea (%)
concepto,,,,,
revenue,10418,2286,1290,1102,99.7
equity,7013,60,41,22,72.7
operating_cash_flow,7349,508,232,155,98.1


### Detalle de ingresos

**Ejemplo de conflicto.** La celda siguiente muestra dos casos reales, del mismo documento:
- **Devon Energy, año 2018.** `RevenueFromContract...` vale 4.449 millones (línea 2, "Revenues") y `Revenues` vale 10.734 millones (línea 4, "Total revenues"). Según el tag elegido, el margen operativo cambia a más del doble. La prioridad elige `Revenues`, que es la línea total.
- **Pioneer Natural Resources, año 2019.** `RevenueFromContract...` vale 9.671 millones (línea 8) y `Revenues` vale 9.304 millones (línea 12). El total es **menor** que la parte: incluye una partida negativa.

`revenue` son los conflictos de ingresos con los dos valores del mismo documento, con la línea (`line`) y el texto (`plabel`) de cada tag en el estado de resultados. Las celdas siguientes lo usan.

In [9]:
# revenue conflicts from one document, with the line and text of each tag in the income statement
is_lines = pre.loc[
    pre["stmt"] == "IS", ["adsh", "tag", "line", "plabel"]
].drop_duplicates(["adsh", "tag"])
revenue_conflicts = conflicts["revenue"]
one_doc = revenue_conflicts.groupby(K)["adsh"].transform("nunique") == 1
revenue = (
    revenue_conflicts[one_doc]
    .merge(is_lines, on=["adsh", "tag"])
    .merge(companies[["cik", "name"]], on="cik")
)


def example(name: str, ddate: int) -> pd.DataFrame:
    """Annual revenue conflict of one company, in millions of USD."""
    rows = revenue[
        revenue["name"].str.startswith(name)
        & (revenue["ddate"] == ddate)
        & (revenue["qtrs"] == 4)
    ]
    return rows.sort_values("line")[["name", "ddate", "tag", "line", "plabel"]].assign(
        millones=rows["value"] / 1e6
    )


pd.concat([example("DEVON", 20181231), example("PIONEER", 20191231)])

,name,ddate,tag,line,plabel,millones
247,DEVON ENERGY CORP/DE,20181231,RevenueFromContractWithCustomerExcludingAssess...,2,Revenues,4449.0
248,DEVON ENERGY CORP/DE,20181231,Revenues,4,Total revenues,10734.0
682,PIONEER NATURAL RESOURCES CO,20191231,RevenueFromContractWithCustomerIncludingAssess...,8,Revenue,9671.0
681,PIONEER NATURAL RESOURCES CO,20191231,Revenues,12,Revenues,9304.0


**Posición y valor de `Revenues` respecto de `RevenueFromContract...`.** `RFC` agrupa los dos tags `RevenueFromContractWithCustomer...` (ingresos por contratos con clientes, norma ASC 606). En cada conflicto con los dos, la tabla cruza dos preguntas:
- ¿La línea de `Revenues` está **antes** o **después** de la de `RFC`?
- ¿El valor de `Revenues` es **mayor** o **menor**?

Cómo se lee:
- **después y mayor:** `Revenues` es el total que suma `RFC` más otros ingresos.
- **después y menor:** `Revenues` es el total, pero incluye una partida negativa, como en Pioneer.
- **antes:** `Revenues` no está en la posición de un total. Son los casos para revisar.

In [10]:
RFC = [
    "RevenueFromContractWithCustomerExcludingAssessedTax",
    "RevenueFromContractWithCustomerIncludingAssessedTax",
]
pairs = revenue[revenue["tag"].isin(["Revenues", *RFC])].assign(
    tag=lambda d: d["tag"].where(d["tag"] == "Revenues", "RFC")
)
# one row per period, with the line and value of Revenues and of RFC
wide = pairs.pivot_table(
    index=K, columns="tag", values=["value", "line"], aggfunc="first"
).dropna()

pd.crosstab(
    (wide["line", "Revenues"] > wide["line", "RFC"]).map(
        {True: "después", False: "antes"}
    ),
    (wide["value", "Revenues"] > wide["value", "RFC"]).map(
        {True: "mayor", False: "menor"}
    ),
    rownames=["línea de Revenues respecto de RFC"],
    colnames=["valor de Revenues respecto de RFC"],
    margins=True,
)

valor de Revenues respecto de RFC,mayor,menor,All
línea de Revenues respecto de RFC,,,
antes,3,0,3
después,772,297,1069
All,775,297,1072


**¿De qué sector son los casos en que `Revenues` es menor?** La celda cuenta esos períodos y sus empresas por sector. La causa probable son pérdidas por coberturas de precio (derivados), pero no se verificó línea por línea.

In [11]:
# periods where the total (Revenues) is smaller than the part (RFC)
smaller = wide[wide["value", "Revenues"] < wide["value", "RFC"]].index.to_frame(
    index=False
)
smaller = smaller.merge(companies[["cik", "sector"]], on="cik")
smaller.groupby("sector").agg(períodos=("cik", "size"), empresas=("cik", "nunique"))

,períodos,empresas
sector,,
energy,297,28


**El texto de la línea de `Revenues`.** Si dice "total", la empresa presenta esa línea como el total de los ingresos.

In [12]:
revenue.loc[revenue["tag"] == "Revenues", "plabel"].str.lower().value_counts().head(15)

plabel
total revenue                                451
total revenues                               262
total operating revenues                     108
total revenues and other income               94
revenues                                      48
total revenues and other                      36
net operating revenues and grant income       23
total operating income                        16
aftermarket services                          12
total                                          8
net operating revenues                         8
total revenue and other                        6
total revenues and other (loss) gain           5
total operating revenues and other income      5
total revenue, net                             5
Name: count, dtype: int64

**Casos en que la prioridad y la última línea difieren.** Se revisan a mano con `plabel`. En estos casos `Revenues` es el total y debajo aparece una línea `RevenueFromContract...` como detalle: se equivoca la regla de la última línea, no la prioridad.

In [13]:
by_priority = revenue.sort_values("priority").drop_duplicates(K).set_index(K)["tag"]
by_last_line = (
    revenue.sort_values("line").drop_duplicates(K, keep="last").set_index(K)["tag"]
)
differ = by_priority[
    by_priority != by_last_line.reindex(by_priority.index)
].index.to_frame(index=False)
revenue.merge(differ, on=K).sort_values([*K, "line"])[
    ["name", "ddate", "qtrs", "tag", "line", "plabel", "value"]
]

,name,ddate,qtrs,tag,line,plabel,value
4,EXPAND ENERGY CORP,20181231,4,Revenues,10,Revenues,1.023100e+10
5,EXPAND ENERGY CORP,20181231,4,RevenueFromContractWithCustomerIncludingAssess...,11,Revenue from contracts with customers,5.189000e+09
1,DENBURY INC,20180630,1,Revenues,2,Revenues,3.870630e+08
3,DENBURY INC,20180630,1,RevenueFromContractWithCustomerExcludingAssess...,13,Revenues,3.822800e+08
0,DENBURY INC,20180630,2,Revenues,2,Revenues,7.402970e+08
2,DENBURY INC,20180630,2,RevenueFromContractWithCustomerExcludingAssess...,13,Revenues,7.298530e+08


**Los conflictos de patrimonio.** En el patrimonio la regla de la última línea no aplica, y la celda siguiente muestra por qué. Son empresas que cambiaron de forma legal en el período, por ejemplo de LLC (`MembersEquity`) o sociedad de socios (`PartnersCapital`) a sociedad anónima (`StockholdersEquity`). El balance muestra la forma vieja y la nueva, a veces con valor 0. Ninguna de las dos líneas es "el total de la otra". `StockholdersEquity` es la forma nueva, por eso va primero.

In [14]:
lines = pre.loc[pre["stmt"] == "BS", ["adsh", "tag", "line", "plabel"]].drop_duplicates(
    ["adsh", "tag"]
)
(
    conflicts["equity"]
    .merge(lines, on=["adsh", "tag"], how="left")
    .merge(companies[["cik", "name"]], on="cik")
    .sort_values([*K, "line"])
    .drop_duplicates(["name", "tag"])[
        ["name", "ddate", "tag", "line", "plabel", "value"]
    ]
)

,name,ddate,tag,line,plabel,value
42,"RILEY EXPLORATION PERMIAN, INC.",20200930,StockholdersEquity,35.0,Total stockholders' equity,4465000.0
47,"RILEY EXPLORATION PERMIAN, INC.",20200930,MembersEquity,40.0,Members' Equity,166617000.0
6,HARVEST OIL & GAS CORP.,20180531,StockholdersEquity,NaN,NaN,248678000.0
38,HARVEST OIL & GAS CORP.,20180531,PartnersCapital,NaN,NaN,0.0
26,PETROTEQ ENERGY INC.,20180831,PartnersCapital,37.0,Share capital,95426796.0
25,PETROTEQ ENERGY INC.,20180831,StockholdersEquity,39.0,Total Shareholders' Equity,32929400.0
17,"ALTA MESA RESOURCES, INC. /DE",20181231,PartnersCapital,59.0,Partners capital,0.0
18,"ALTA MESA RESOURCES, INC. /DE",20181231,StockholdersEquity,64.0,Total stockholders equity/partners capital,-29393000.0
0,LIBERTY ENERGY INC.,20180331,MembersEquity,39.0,Member equity,0.0
1,LIBERTY ENERGY INC.,20180331,StockholdersEquity,44.0,Total stockholders' equity,372822000.0


## Prueba 4: los alternativos de la guía de la profesora

La guía propone dos alternativos que no están en la lista:
- `StockholdersEquityAttributableToParent` para el patrimonio. La celda cuenta sus filas en `num.csv`.
- `ProfitLoss` para la ganancia neta. `ProfitLoss` incluye la parte de los socios minoritarios (las participaciones no controladoras); `NetIncomeLoss` no. Si la diferencia entre los dos es justamente `NetIncomeLossAttributableToNoncontrollingInterest` (la parte de los minoritarios), `ProfitLoss` no es un sustituto de `NetIncomeLoss`.

La celda mide, en los períodos con los dos tags, cuántas veces difieren y cuántas de esas la diferencia es la parte de los minoritarios.

In [15]:
print(
    "filas con StockholdersEquityAttributableToParent:",
    (num["tag"] == "StockholdersEquityAttributableToParent").sum(),
)

net = values[["NetIncomeLoss", "ProfitLoss"]].dropna()
minority = col("NetIncomeLossAttributableToNoncontrollingInterest").reindex(net.index)
gap = net["ProfitLoss"] - net["NetIncomeLoss"]

differ = gap.abs() > 0.01 * net["NetIncomeLoss"].abs()
# the difference equals the minority partners' part (within 1% of ProfitLoss)
explained = (gap - minority).abs() <= 0.01 * net["ProfitLoss"].abs()
pd.Series(
    {
        "períodos con NetIncomeLoss y ProfitLoss": len(net),
        "difieren más de 1%": differ.sum(),
        "de esos, la diferencia es la parte de los minoritarios": (differ & explained).sum(),
    }
)

filas con StockholdersEquityAttributableToParent: 0


períodos con NetIncomeLoss y ProfitLoss                   4825
difieren más de 1%                                        2183
de esos, la diferencia es la parte de los minoritarios    1803
dtype: int64

## Prueba 5: las construcciones

Una **construcción** calcula un concepto con otros tags cuando la empresa no reporta el tag del concepto. Ejemplo: pasivo = (pasivo + patrimonio) − patrimonio. Antes de usarla, hay que comprobar que da el mismo valor que el tag, en las empresas que reportan las dos cosas.

`check(tag, construcción)` devuelve:
- **períodos con tag y construcción:** períodos en que existen los dos valores. Es la base de la comparación.
- **coincide (%):** porcentaje de esos períodos en que la construcción difiere del tag en 1% o menos.
- **empresas sin el tag:** empresas que nunca reportan el tag en 2018–2023.
- **recuperadas:** empresas sin el tag que tienen la construcción en al menos un período.

In [16]:
def check(tag: str, built: pd.Series) -> dict:
    """Compares a construction with the reported tag, and counts the companies it recovers."""
    both = pd.concat({"tag": col(tag), "built": built}, axis=1).dropna()
    match = (both["built"] - both["tag"]).abs() <= 0.01 * both["tag"].abs()

    # companies that never report the tag, but have the construction in some period
    without_tag = set(num["cik"]) - set(num.loc[num["tag"] == tag, "cik"])
    recovered = without_tag & set(built.dropna().index.get_level_values("cik"))
    sector = companies.set_index("cik").loc[list(recovered), "sector"].value_counts()
    return {
        "períodos con tag y construcción": len(both),
        "coincide (%)": round(match.mean() * 100, 1),
        "empresas sin el tag": len(without_tag),
        "recuperadas": len(recovered),
        "recuperadas energía": sector.get("energy", 0),
        "recuperadas salud": sector.get("health", 0),
    }

Las construcciones que se comparan (las filas "solo" o sin un término son variantes peores, para mostrar por qué no se usan):

| Concepto | Construcción | Tags |
|---|---|---|
| Resultado operativo | ingresos − costos y gastos totales | ingresos (con la lista de `CONCEPTS`) − `CostsAndExpenses` |
| Resultado operativo (variante) | ingresos − gastos operativos | ingresos − `OperatingExpenses` |
| Ganancia neta | ganancia total − parte de los minoritarios | `ProfitLoss` − `NetIncomeLossAttributableToNoncontrollingInterest` |
| Ganancia neta (variante) | ganancia total | `ProfitLoss` |
| Activo | pasivo + patrimonio | `LiabilitiesAndStockholdersEquity` |
| Pasivo | (pasivo + patrimonio) − patrimonio total − capital temporario | `LiabilitiesAndStockholdersEquity` − `total_equity` − `temporary_equity` |
| Pasivo (variante) | sin restar el capital temporario | `LiabilitiesAndStockholdersEquity` − `total_equity` |
| Patrimonio | patrimonio con minoritarios − parte de los minoritarios | `StockholdersEquityIncludingPortionAttributableToNoncontrollingInterest` − `MinorityInterest` |
| Patrimonio de una sociedad de socios | capital de socios con minoritarios − parte de los minoritarios | `PartnersCapitalIncludingPortionAttributableToNoncontrollingInterest` − `MinorityInterest` |

- `total_equity`: el patrimonio **con** los minoritarios, porque `LiabilitiesAndStockholdersEquity` los incluye. Se toma el primer tag con valor entre las formas legales: sociedad anónima con minoritarios, sociedad anónima, sociedad de socios con minoritarios, sociedad de socios, LLC.
- `temporary_equity`: el capital temporario, acciones que la empresa puede tener que recomprar. No es pasivo ni patrimonio, pero está dentro de `LiabilitiesAndStockholdersEquity`. Si falta, vale 0.
- Si falta la parte de los minoritarios, vale 0: la empresa no tiene minoritarios.

In [17]:
revenue = first(CONCEPTS["revenue"])
minority_income = col("NetIncomeLossAttributableToNoncontrollingInterest").fillna(0)
total_equity = first(
    [
        "StockholdersEquityIncludingPortionAttributableToNoncontrollingInterest",
        "StockholdersEquity",
        "PartnersCapitalIncludingPortionAttributableToNoncontrollingInterest",
        "PartnersCapital",
        "MembersEquity",
    ]
)
temporary_equity = sum(
    col(t).fillna(0)
    for t in [
        "TemporaryEquityCarryingAmountAttributableToParent",
        "RedeemableNoncontrollingInterestEquityCarryingAmount",
    ]
)
liabilities_and_equity = col("LiabilitiesAndStockholdersEquity")

constructions = {
    ("operating_income", "ingresos − CostsAndExpenses"): check(
        "OperatingIncomeLoss", revenue - col("CostsAndExpenses")
    ),
    ("operating_income", "ingresos − OperatingExpenses"): check(
        "OperatingIncomeLoss", revenue - col("OperatingExpenses")
    ),
    ("net_income", "ProfitLoss − minoritarios"): check(
        "NetIncomeLoss", col("ProfitLoss") - minority_income
    ),
    ("net_income", "ProfitLoss solo"): check("NetIncomeLoss", col("ProfitLoss")),
    ("assets", "LiabilitiesAndStockholdersEquity"): check(
        "Assets", liabilities_and_equity
    ),
    (
        "liabilities",
        "pasivo + patrimonio − patrimonio total − capital temporario",
    ): check("Liabilities", liabilities_and_equity - total_equity - temporary_equity),
    ("liabilities", "pasivo + patrimonio − patrimonio total"): check(
        "Liabilities", liabilities_and_equity - total_equity
    ),
    ("equity", "patrimonio con minoritarios − MinorityInterest"): check(
        "StockholdersEquity",
        col("StockholdersEquityIncludingPortionAttributableToNoncontrollingInterest")
        - col("MinorityInterest").fillna(0),
    ),
    ("equity", "capital de socios con minoritarios − MinorityInterest"): check(
        "PartnersCapital",
        col("PartnersCapitalIncludingPortionAttributableToNoncontrollingInterest")
        - col("MinorityInterest").fillna(0),
    ),
}
pd.DataFrame(constructions).T.rename_axis(["concepto", "construcción"])

períodos con tag y construcción  \
concepto         construcción                                                                          
operating_income ingresos − CostsAndExpenses                                                  3875.0   
                 ingresos − OperatingExpenses                                                 3846.0   
net_income       ProfitLoss − minoritarios                                                    4825.0   
                 ProfitLoss solo                                                              4825.0   
assets           LiabilitiesAndStockholdersEquity                                             6856.0   
liabilities      pasivo + patrimonio − patrimonio total − capita...                           5330.0   
                 pasivo + patrimonio − patrimonio total                                       5330.0   
equity           patrimonio con minoritarios − MinorityInterest                               2263.0   
                 capital de socios con minoritarios − MinorityIn...                             74.0   

                                                                     coincide (%)  \
concepto         construcción                                                       
operating_income ingresos − CostsAndExpenses                                 89.7   
                 ingresos − OperatingExpenses                                70.2   
net_income       ProfitLoss − minoritarios                                   91.2   
                 ProfitLoss solo                                             54.8   
assets           LiabilitiesAndStockholdersEquity                           100.0   
liabilities      pasivo + patrimonio − patrimonio total − capita...          94.7   
                 pasivo + patrimonio − patrimonio total                      86.2   
equity           patrimonio con minoritarios − MinorityInterest              93.1   
                 capital de socios con minoritarios − MinorityIn...          23.0   

                                                                     empresas sin el tag  \
concepto         construcción                                                              
operating_income ingresos − CostsAndExpenses                                        55.0   
                 ingresos − OperatingExpenses                                       55.0   
net_income       ProfitLoss − minoritarios                                          11.0   
                 ProfitLoss solo                                                    11.0   
assets           LiabilitiesAndStockholdersEquity                                    2.0   
liabilities      pasivo + patrimonio − patrimonio total − capita...                 90.0   
                 pasivo + patrimonio − patrimonio total                             90.0   
equity           patrimonio con minoritarios − MinorityInterest                     45.0   
                 capital de socios con minoritarios − MinorityIn...                445.0   

                                                                     recuperadas  \
concepto         construcción                                                      
operating_income ingresos − CostsAndExpenses                                34.0   
                 ingresos − OperatingExpenses                               10.0   
net_income       ProfitLoss − minoritarios                                  10.0   
                 ProfitLoss solo                                            10.0   
assets           LiabilitiesAndStockholdersEquity                            0.0   
liabilities      pasivo + patrimonio − patrimonio total − capita...         89.0   
                 pasivo + patrimonio − patrimonio total                     89.0   
equity           patrimonio con minoritarios − MinorityInterest             11.0   
                 capital de socios con minoritarios − MinorityIn...          3.0   

                                                  

## Sección 6: ratios descartados

Cobertura de los tags de los dos ratios descartados (margen bruto e I+D / ingresos): empresas que reportan el tag al menos una vez, y su porcentaje sobre el total del sector.

In [18]:
total = num.groupby("sector")["cik"].nunique()
discarded = num[num["tag"].isin(["GrossProfit", "ResearchAndDevelopmentExpense"])]
count = discarded.groupby(["tag", "sector"])["cik"].nunique().unstack()
pd.concat({"empresas": count, "% del sector": (count / total * 100).round(1)}, axis=1)

empresas        % del sector       
sector                          energy health       energy health
tag                                                              
GrossProfit                         37     62         12.1   35.2
ResearchAndDevelopmentExpense       14     51          4.6   29.0

En salud, el gasto en I+D (investigación y desarrollo) por código SIC. "con I+D" son las empresas que reportan `ResearchAndDevelopmentExpense` al menos una vez. El SIC 8071 son los laboratorios médicos.

In [19]:
# health: companies with R&D by SIC code
health = companies[
    (companies["sector"] == "health") & companies["cik"].isin(num["cik"])
].copy()
health["con I+D"] = health["cik"].isin(
    num.loc[num["tag"] == "ResearchAndDevelopmentExpense", "cik"]
)
health.groupby("sic")["con I+D"].agg(["sum", "count"]).rename(
    columns={"sum": "con I+D", "count": "empresas"}
)

,con I+D,empresas
sic,,
8000,8,33
8011,1,13
8050,0,9
8051,0,7
8060,2,6
8062,0,13
8071,27,44
8082,0,17
8090,11,26


## Sección 7: HCA Healthcare

`docs/ratios.md` usa a HCA como ejemplo de patrimonio negativo en el ROE. La celda lee `num.csv` **sin** el filtro de 2018–2023, porque los primeros documentos traen comparativos de 2016 y 2017. Muestra `StockholdersEquity` al cierre de cada año, en millones de USD.

In [20]:
# all the dates of num.csv, also before 2018 (comparatives of the first documents)
everything = pd.read_csv("data/num.csv", dtype={"cik": str})
hca = companies.loc[companies["name"].str.startswith("HCA"), "cik"].iloc[0]
(
    everything[
        (everything["cik"] == hca)
        & (everything["tag"] == "StockholdersEquity")
        & (everything["ddate"] % 10000 == 1231)
    ]
    .drop_duplicates("ddate")
    .set_index("ddate")["value"]
    .div(1e6)
    .rename("StockholdersEquity al 31/12 (millones de USD)")
)

ddate
20161231   -7302.0
20171231   -6806.0
20181231   -4950.0
20191231   -2808.0
20201231     572.0
20211231    -933.0
20221231   -2767.0
20231231   -1774.0
Name: StockholdersEquity al 31/12 (millones de USD), dtype: float64

## Resultado

Cada número de `docs/ratios.md` que no es una cobertura por tag, con la celda que lo calcula. "Coincide" significa diferencia de 1% o menos.

**Listas de tags (`concepts.py`)**

| Número | Valor | Celda |
|---|---|---|
| Conceptos con un tag de la lista en el estado en más del 95% de los documentos | 8 de 9 | `cell-08` |
| Ganancia neta: tag de la lista en el estado | 91,6% | `cell-08` |
| Empresas de energía con `PartnersCapital` | 38 | `cell-12` |
| Períodos en que se elige `SalesRevenueGoodsNet` | 0 | `cell-12` |
| Períodos en que se eligen los tags 4 a 10 de ingresos | 0 a 15 | `cell-12` |
| Ingresos: períodos / con más de un tag / conflictos | 10.418 / 2.286 / 1.290 | `cell-14` |
| Patrimonio: períodos / con más de un tag / conflictos | 7.013 / 60 / 41 | `cell-14` |
| Flujo de caja operativo: períodos / con más de un tag / conflictos | 7.349 / 508 / 232 | `cell-14` |
| Prioridad = última línea: ingresos / flujo de caja operativo / patrimonio | 99,7% / 98,1% / 72,7% | `cell-14` |
| Ingresos: conflictos del mismo documento | 1.102 | `cell-14` |
| Devon 2018: `RFC` / `Revenues` | 4.449 / 10.734 millones de USD | `cell-16` |
| Pioneer 2019: `RFC` / `Revenues` | 9.671 / 9.304 millones de USD | `cell-16` |
| Pares `Revenues`–`RFC` en que `Revenues` es menor | 297 de 1.072 (28%) | `cell-18` |
| Esos casos: sector y empresas | energía, 28 empresas | `cell-20` |
| Texto de la línea de `Revenues` | casi siempre "total revenue(s)" | `cell-22` |
| Casos en que la prioridad y la última línea difieren | 3 (Expand Energy y Denbury) | `cell-24` |
| Filas de `StockholdersEquityAttributableToParent` | 0 | `cell-28` |
| `ProfitLoss` difiere de `NetIncomeLoss` / la diferencia es la parte de los minoritarios | 2.183 / 1.803 (83%) | `cell-28` |

**Construcciones**

| Construcción | Coincide | Recupera | Celda |
|---|---|---|---|
| Resultado operativo = ingresos − `CostsAndExpenses` | 89,7% | 34 de 55 empresas | `cell-32` |
| Resultado operativo = ingresos − `OperatingExpenses` | 70,2% | — | `cell-32` |
| Ganancia neta = `ProfitLoss` − minoritarios | 91,2% | 10 de 11 | `cell-32` |
| Ganancia neta = `ProfitLoss` solo | 54,8% | — | `cell-32` |
| Activo = `LiabilitiesAndStockholdersEquity` | 100% | 0 de 2 | `cell-32` |
| Pasivo = pasivo + patrimonio − patrimonio total − capital temporario | 94,7% | 89 de 90 | `cell-32` |
| Pasivo, sin restar el capital temporario | 86,2% | — | `cell-32` |
| Patrimonio = patrimonio con minoritarios − `MinorityInterest` | 93,1% | 11 de 45 | `cell-32` |
| Capital de socios = capital con minoritarios − `MinorityInterest` | 23,0% (74 períodos): no sirve | 3 de 445 | `cell-32` |

**Otros números**

| Número | Valor | Celda |
|---|---|---|
| Energía con `GrossProfit` | 37 de 307 (12,1%) | `cell-34` |
| Energía con `ResearchAndDevelopmentExpense` | 14 de 307 (4,6%) | `cell-34` |
| Salud, SIC 8071, con I+D | 27 de 44 | `cell-36` |
| HCA: `StockholdersEquity` al 31/12/2018 | −4.950 millones de USD | `cell-38` |
| HCA: años con patrimonio negativo entre 2016 y 2023 | 7 de 8 (positivo solo en 2020) | `cell-38` |